In [4]:
from result_exploration.analysis import plot_predictions
import pandas as pd
from result_exploration.analysis import EXPERIMENT_CONFIGS
import ipywidgets as widgets
from IPython.display import display
from result_exploration.analysis import plot_predictions, EXPERIMENT_CONFIGS, MODEL_ORDER, BENCHMARKS, AGG_LEVEL_ORDER, CONFIG_META

In [5]:
rows = [
    {"config": cfg, "model": model, "run": path.name}
    for cfg, models in EXPERIMENT_CONFIGS.items()
    for model, path in models.items()
]
pd.DataFrame(rows).pivot(index="model", columns="config", values="run")

config,h_168_24,h_744_168,m_1008_144,m_288_144
model,,,,
chronos,20260426_200553_chronos_h_168_24,20260426_203825_chronos_h_744_168,20260426_213125_chronos_m_1008_144,20260426_205531_chronos_m_288_144
gru,20260426_200957_gru_h_168_24,20260426_204352_gru_h_744_168,20260426_214707_gru_m_1008_144,20260426_210023_gru_m_288_144
gru_fcn,20260426_202146_gru_fcn_h_168_24,20260426_204341_gru_fcn_h_744_168,20260426_221256_gru_fcn_m_1008_144,20260426_205908_gru_fcn_m_288_144
lstm,20260426_202318_lstm_h_168_24,20260426_204900_lstm_h_744_168,20260426_215602_lstm_m_1008_144,20260426_205905_lstm_m_288_144
moirai,20260426_202404_moirai_h_168_24,20260426_203757_moirai_h_744_168,20260426_214426_moirai_m_1008_144,20260426_210023_moirai_m_288_144
prophet,20260426_200640_prophet_h_168_24,20260428_114207_prophet_h_744_168,20260426_201346_prophet_m_1008_144,20260426_200823_prophet_m_288_144
sarima,20260426_200640_sarima_h_168_24,20260324_225714_sarima_h_744_168,20260428_115305_sarima_m_1008_144,20260426_200824_sarima_m_288_144
seasonal_naive,20260428_113720_seasonal_naive_h_168_24,20260426_200640_seasonal_naive_h_744_168,20260428_113904_seasonal_naive_m_1008_144,20260426_200823_seasonal_naive_m_288_144
timesfm,20260426_200640_timesfm_h_168_24,20260426_203757_timesfm_h_744_168,20260426_210812_timesfm_m_1008_144,20260426_204458_timesfm_m_288_144


In [6]:
def get_series_ids(config_name, bench, model, agg_level):
    runs = EXPERIMENT_CONFIGS[config_name]
    if model not in runs:
        return []
    run_dir = runs[model]
    run_subdirs = sorted(d for d in run_dir.iterdir() if d.is_dir() and d.name.startswith("run_"))
    if not run_subdirs:
        return []
    pred = run_subdirs[0] / f"{bench}_predictions.csv"
    if not pred.exists():
        return []
    df = pd.read_csv(pred, usecols=["id", "agg_level"])
    return sorted(df[df["agg_level"] == agg_level]["id"].unique().tolist())

config_dd  = widgets.Dropdown(options=list(EXPERIMENT_CONFIGS.keys()), description="Config:")
bench_dd   = widgets.Dropdown(options=BENCHMARKS, description="Benchmark:")
model_dd   = widgets.Dropdown(options=MODEL_ORDER, description="Model:")
agg_dd     = widgets.Dropdown(options=AGG_LEVEL_ORDER, description="Agg level:")
series_dd  = widgets.Dropdown(options=[], description="Series ID:")
out = widgets.Output()

def update_series(*_):
    ids = get_series_ids(config_dd.value, bench_dd.value, model_dd.value, agg_dd.value)
    series_dd.options = ids
    if ids:
        series_dd.value = ids[0]

for w in [config_dd, bench_dd, model_dd, agg_dd]:
    w.observe(update_series, names="value")

update_series()

btn = widgets.Button(description="Plot", button_style="primary")

def on_plot(_):
    out.clear_output()
    with out:
        try:
            freq = CONFIG_META[config_dd.value]["freq"]
            fig = plot_predictions(
                bench=bench_dd.value,
                model=model_dd.value,
                series_id=series_dd.value,
                agg_level=agg_dd.value,
                runs=EXPERIMENT_CONFIGS[config_dd.value],
                freq=freq,
            )
            fig.show()
        except Exception as e:
            print(f"Error: {e}")

btn.on_click(on_plot)
display(widgets.VBox([widgets.HBox([config_dd, bench_dd, model_dd, agg_dd, series_dd]), btn, out]))